<a href="https://colab.research.google.com/github/MalikZeeshan1122/FlyRank-ML-Internship-Starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MalikZeeshan1122/FlyRank-ML-Internship-Starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
import os

repo_path = "/content/flyrank-ml-internship-starter"

if not os.path.exists(repo_path):
    !git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

print("Repository exists:", os.path.exists(repo_path))

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 299, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 299 (delta 130), reused 98 (delta 98), pack-reused 106 (from 1)
Receiving objects: 100% (299/299), 1.88 MiB | 3.66 MiB/s, done.
Resolving deltas: 100% (161/161), done.
Repository exists: True


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd

# Find the starter dataset
possible_paths = [
    "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv",
    "flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
]

csv_path = next(
    (path for path in possible_paths if os.path.exists(path)),
    None
)

if csv_path is None:
    raise FileNotFoundError(
        "Starter CSV not found. Make sure the starter repository "
        "is available in the Colab runtime."
    )

df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("Number of fields:", len(df.columns))
print("\nFields:")
print(df.columns.tolist())


Dataset shape: (30000, 44)
Number of fields: 44

Fields:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [6]:
# 2. Feature notes

# Identify numeric and categorical fields
numeric_features = df.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = [
    col for col in df.columns
    if col not in numeric_features
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nNumeric feature missing values:")
print(df[numeric_features].isna().sum().sort_values(ascending=False).head(10))

print("\nCategorical feature missing values:")
if categorical_features:
    print(
        df[categorical_features]
        .isna()
        .sum()
        .sort_values(ascending=False)
        .head(10)
    )
else:
    print("No categorical features found.")

print("\nFeature availability rule:")
print("Features must be available before the prediction point.")

Numeric features: 30
Categorical features: 14

Numeric feature missing values:
char_count       7699
word_count       7699
trend_pct        3388
competition      2468
cpc              2468
search_volume    2468
scroll_rate       125
pageviews_90d       0
clicks_90d          0
users_90d           0
dtype: int64

Categorical feature missing values:
provider_used        21438
word_count_tier       7699
char_count_tier       7699
model_used            5733
competition_level     2610
main_intent           2374
content_type             0
client_id                0
content_id               0
age_tier                 0
dtype: int64

Feature availability rule:
Features must be available before the prediction point.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 3. Leakage hunt

# Known label-derived fields
label_derived = [
    "trend_direction",
    "trend_pct"
]

# Look for fields that may contain future-window information
future_keywords = [
    "future",
    "post",
    "after",
    "next",
    "later"
]

# Look for product-related fields that may need review
product_keywords = [
    "product",
    "plan",
    "package",
    "tier"
]

# Check label-derived fields
found_label_leakage = [
    col for col in label_derived
    if col in df.columns
]

# Check possible future fields
found_future_fields = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in future_keywords)
]

# Check possible product fields
found_product_fields = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in product_keywords)
]

print("Label-derived fields found:")
print(found_label_leakage)

print("\nPossible future-window fields:")
print(found_future_fields)

print("\nPossible product-related fields:")
print(found_product_fields)

print("\nLeakage hunt completed.")

Label-derived fields found:
['trend_direction', 'trend_pct']

Possible future-window fields:
[]

Possible product-related fields:
['age_tier', 'age_tier_order', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']

Leakage hunt completed.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 4. What I excluded and why

excluded_fields = {
    "trend_direction": "Label-derived outcome; leakage risk.",
    "trend_pct": "Outcome-derived trend measure; leakage risk.",
    "client_hash_id": "Identifier; not a meaningful predictive feature.",
    "content_hash_id": "Identifier; not a meaningful predictive feature.",
}

print("Excluded fields and reasons:")

for field, reason in excluded_fields.items():
    if field in df.columns:
        print(f"- {field}: EXCLUDED — {reason}")
    else:
        print(f"- {field}: Not present in dataset")


Excluded fields and reasons:
- trend_direction: EXCLUDED — Label-derived outcome; leakage risk.
- trend_pct: EXCLUDED — Outcome-derived trend measure; leakage risk.
- client_hash_id: Not present in dataset
- content_hash_id: Not present in dataset


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.